# K-Means Clustering — From Scratch
> Unsupervised. Iteratively assigns points to nearest centroid and updates centroids.

Converges when centroids stop moving.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
np.random.seed(42)

## KMeans Class

In [ ]:
class KMeans:
    def __init__(self, k=3, max_iters=300, tol=1e-4):
        self.k = k; self.max_iters = max_iters; self.tol = tol

    def fit(self, X):
        # Random initialisation
        idx = np.random.choice(len(X), self.k, replace=False)
        self.centroids = X[idx].copy()
        self.inertia_history = []

        for _ in range(self.max_iters):
            labels = self._assign(X)
            new_c  = np.array([X[labels==k].mean(axis=0) if (labels==k).any()
                                else self.centroids[k] for k in range(self.k)])
            self.inertia_history.append(self._inertia(X, labels))
            if np.linalg.norm(new_c - self.centroids) < self.tol:
                break
            self.centroids = new_c
        self.labels_ = labels

    def _assign(self, X):
        dists = np.array([[np.linalg.norm(x - c) for c in self.centroids] for x in X])
        return np.argmin(dists, axis=1)

    def _inertia(self, X, labels):
        return sum(np.sum((X[labels==k] - self.centroids[k])**2) for k in range(self.k))

    def predict(self, X):
        return self._assign(X)

## Fit & Visualise

In [ ]:
X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)

km = KMeans(k=4)
km.fit(X)

colors = plt.cm.tab10(np.linspace(0, 0.5, 4))
plt.figure(figsize=(7,5))
for k in range(4):
    pts = X[km.labels_==k]
    plt.scatter(pts[:,0], pts[:,1], s=30, alpha=0.7, label=f"Cluster {k}")
plt.scatter(km.centroids[:,0], km.centroids[:,1],
            c='black', marker='X', s=200, zorder=5, label='Centroids')
plt.title("K-Means Clustering"); plt.legend(); plt.tight_layout(); plt.show()

## Elbow Method – Find Optimal K

In [ ]:
inertias = []
ks = range(1, 11)
for k in ks:
    m = KMeans(k=k); m.fit(X)
    inertias.append(m.inertia_history[-1])

plt.figure(figsize=(8,4))
plt.plot(ks, inertias, 'o-', color='steelblue')
plt.xlabel("Number of Clusters (K)"); plt.ylabel("Inertia")
plt.title("Elbow Method"); plt.grid(True); plt.tight_layout(); plt.show()

## Convergence of Centroids

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(km.inertia_history, color='tomato')
plt.xlabel("Iteration"); plt.ylabel("Inertia")
plt.title("K-Means – Inertia per Iteration"); plt.tight_layout(); plt.show()